In [1]:
import re
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

ds = load_dataset("QCRI/HumAID-all", verification_mode="no_checks")
df = ds["train"].to_pandas()
print(df.shape)

/home/imagine29/green-ai-compression-study/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(53531, 2)


In [2]:
import re

def clean_tweet(text):
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    text = re.sub(r'^RT\s+@\w+:\s*', '', text)
    return text.strip()

df["text_clean"] = df["tweet_text"].apply(clean_tweet)

In [3]:
print("Before dedup:", len(df))
df = df.drop_duplicates(subset="text_clean")
print("After dedup:", len(df))

Before dedup: 53531
After dedup: 53530


In [4]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["class_label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["class_label"], random_state=42)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 37471 Val: 8029 Test: 8030


In [5]:
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name}:")
    print(split["class_label"].value_counts(normalize=True))


train:
class_label
rescue_volunteering_or_donation_effort    0.278188
other_relevant_information                0.158816
sympathy_and_support                      0.116730
infrastructure_and_utility_damage         0.106749
injured_or_dead_people                    0.095460
not_humanitarian                          0.082330
caution_and_advice                        0.070508
displaced_people_and_evacuations          0.052307
requests_or_urgent_needs                  0.034240
missing_or_found_people                   0.004670
Name: proportion, dtype: float64

val:
class_label
rescue_volunteering_or_donation_effort    0.278117
other_relevant_information                0.158799
sympathy_and_support                      0.116702
infrastructure_and_utility_damage         0.106738
injured_or_dead_people                    0.095529
not_humanitarian                          0.082327
caution_and_advice                        0.070494
displaced_people_and_evacuations          0.052310
requests_or

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

for sample in train_df["text_clean"].iloc[:5]:
    print(sample)
    print(tokenizer.tokenize(sample), "\n")

JUST IN: Weve been officially deployed by @FEMA to assist with Hurricane #Harvey aftermath. @MedCenterAir is sending 2 planes to TX today.
['JUST', 'ĠIN', ':', 'ĠWe', 've', 'Ġbeen', 'Ġofficially', 'Ġdeployed', 'Ġby', 'Ġ@', 'F', 'EMA', 'Ġto', 'Ġassist', 'Ġwith', 'ĠHurricane', 'Ġ#', 'Har', 'vey', 'Ġaftermath', '.', 'Ġ@', 'Med', 'Center', 'Air', 'Ġis', 'Ġsending', 'Ġ', '2', 'Ġplanes', 'Ġto', 'ĠTX', 'Ġtoday', '.'] 

Also you can donate to the @topos rescue brigade via PayPal: donativos@brigada-rescate-topos.org ὤF἟2἟D
['Also', 'Ġyou', 'Ġcan', 'Ġdonate', 'Ġto', 'Ġthe', 'Ġ@', 'top', 'os', 'Ġrescue', 'Ġbrigade', 'Ġvia', 'ĠPayPal', ':', 'Ġdon', 'ativos', '@', 'brig', 'ada', '-res', 'cate', '-top', 'os', '.org', 'Ġá', '½', '¤', 'F', 'á¼', 'Ł', '2', 'á¼', 'Ł', 'D'] 

Training at Premier Martial Arts Cranston takes children to the next level of focus and determination that they need to be successful in school, life and karate! Call today and receive 2
['Training', 'Ġat', 'ĠPremier', 'ĠMartial', '

In [7]:
train_df["token_length"] = train_df["text_clean"].apply(lambda x: len(tokenizer.tokenize(x)))
print(train_df["token_length"].describe())

count    37471.000000
mean        34.433135
std         18.778090
min          3.000000
25%         21.000000
50%         30.000000
75%         44.000000
max        397.000000
Name: token_length, dtype: float64


In [8]:
worst = train_df.loc[train_df["token_length"].idxmax()]
print(worst["tweet_text"])
print(worst["text_clean"])
print(worst["token_length"])

@RaghupathiBhat @Meenu_71 @HarishK04131926 @BJP4Udupi @mattarhegde @UdayKumarBJP @YashpalBJP @DheerajGbc @GogoiRanju @ajaykumar2697 @Harvansh_Batra @VictoryForNamo @AmitShahKiSena @imShefalii @iSanjuktaP @prayag @BillionIndian @LillyMaryPinto @narendramodi @sukanyaiyer2 @shitijsrivastav @Babble524 @draksbond @DrSarojiniMLC @NazlinShaikh @PathanAsmakhan @kapil9994 @pooja303singh @AmitShah @bakoriya_kirti_ @narendrap85 @YadavAnkes @sbhandari16 @DebashishHiTs @BengaluruBullet @meenakshisharan @shakthigj @shefalitiwari7 @truevirathindu @Dubey1Vij @rajasve @sanghavideepa @BetiBachaoBetiP @MinistryWCD @dr_maheshsharma @moefcc @drharshvardhan @nitin_gadkari @raosahebdanve #NH66 Near Pangala (b/w #Udupi #Mangaluru) #flood water has began to flow over the #HighWay. MLA @lalajibjp ji visited just now. Rescue Teams hav arrived. Water level is raising minute by minute. Same situation near #Padubidri too. Be careful while driving. #Rain #floods
@RaghupathiBhat @Meenu_71 @HarishK04131926 @BJP4Udupi 

In [9]:
print(train_df["token_length"].quantile([0.90, 0.95, 0.99]))

0.90    61.0
0.95    69.0
0.99    86.0
Name: token_length, dtype: float64


In [10]:
import os
os.makedirs("data/processed", exist_ok=True)
train_df.to_csv("data/processed/train.csv", index=False)
val_df.to_csv("data/processed/val.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)